# langchain-arabic quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/louaychoum/langchain-arabic/blob/v0.4.0/examples/quickstart.ipynb)

Process Arabic text and a fake chat-model response without API keys. The fake model and processing run locally. The install cell needs network access and the 0.4.0 release on PyPI; the Colab badge needs the pushed version tag. Python 3.10+ is recommended for current LangChain.


In [ ]:
%pip -q install "langchain-arabic==0.4.0"

## A chat-model chain
Only the supplied dictionary words receive diacritics.


In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

from langchain_arabic import ArabicTextOutputParser

raw = "هذه شركة تقنية. نسبة النمو: 95%."
llm = FakeListChatModel(responses=[raw])
parser = ArabicTextOutputParser(diacritics_map={"شركة": "شَرِكَة", "تقنية": "تِقْنِيَّة"})
chain = llm | parser
expected = "هذه شَرِكَة تِقْنِيَّة. نسبة النمو: خمسة و تسعون بالمائة."
result = chain.invoke("كيف كان النمو؟")
print("Raw:      ", raw)
print("Processed:", result)
assert result == expected

## Standalone numbers and diacritics
Decimal digits preserve precision; the surrounding text and punctuation survive.


In [ ]:
from langchain_arabic import DiacriticsProcessor, convert_numbers_in_text

assert convert_numbers_in_text("١٢٫٥٠ ريال،") == "اثنا عشر فاصلة خمسة صفر ريال،"
assert convert_numbers_in_text("1000000") == "مليون"
assert convert_numbers_in_text("-0.050", "en") == "minus zero point zero five zero"
fixes = DiacriticsProcessor("تقنية -> تِقْنِيَّة")
assert fixes.process("تَقْنِيَة") == "تِقْنِيَّة"
print(convert_numbers_in_text("١٢٫٥٠ ريال،"))

## Inspectable speech preparation
Use the opt-in normalizer for supported dates, times, identifiers and amounts. Keep the original text for display. These rules do not guarantee better audio; ambiguous dates are retained unless you choose a date order.


In [ ]:
from langchain_arabic import ArabicSpeechNormalizer

normalizer = ArabicSpeechNormalizer()
speech_raw = "رقم الطلب 001234. المبلغ ١٢٫٥٠ SAR."
speech = normalizer.explain(speech_raw)
assert (
    speech.text
    == "رقم الطلب صفر صفر واحد اثنان ثلاثة أربعة. المبلغ اثنا عشر فاصلة خمسة صفر ريال سعودي."
)
assert normalizer.process("03/04/2026") == "03/04/2026"
print(speech.text)
for change in speech.changes:
    assert speech_raw[change.start : change.end] == change.original
    print(change.kind, change.original, change.replacement)

## Gulf number substitutions and selected contexts
Gulf support is a small set of substitutions, not full dialect grammar.


In [ ]:
gulf = ArabicTextOutputParser.for_tts_fixes({"عداد": "عَدَّاد"}, dialect="gulf")
assert gulf.parse("عداد 100%") == "عَدَّاد مية بالمية"
percent_only = ArabicTextOutputParser(number_contexts={"percentage"})
assert percent_only.parse("95% من 500") == "خمسة و تسعون بالمائة من 500"
print(gulf.parse("عداد 100%"))

## Async and buffered streaming
The parser yields one complete answer after the model finishes, in both sync and async streams.


In [ ]:
assert list(chain.stream("كيف كان النمو؟")) == [expected]
assert await chain.ainvoke("كيف كان النمو؟") == expected
chunks = [chunk async for chunk in chain.astream("كيف كان النمو؟")]
assert chunks == [expected]
print(chunks)

## Limits and optional neural inference

Dictionary mode does not infer unknown diacritics or grammatical agreement. Number conversion is not a general identifier/date parser. The optional `catt` backend may download weights and requires an upstream-supported ONNX GPU platform; it is not run in this notebook. The project's small MIT-licensed evaluation set is a smoke check, not general accuracy evidence.

See [README](https://github.com/louaychoum/langchain-arabic#readme) and [evaluation methodology](https://github.com/louaychoum/langchain-arabic/tree/v0.4.0/benchmarks).
